In [2]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 10.2 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import LeaveOneGroupOut
from scipy.spatial import cKDTree

In [3]:
df = pd.read_csv('/content/grid_features.csv')
df.head()

,lat,lon,city,atm_count,sber_count,tinkoff_count,vtb_count,alfa_count,gazprom_count,raiff_count,...,poi_diversity_500m,poi_diversity_1000m,total_poi_500m,total_poi_1000m,retail_share_500m,residential_ratio_500m,orgs_500m,org_diversity_500m,orgs_1000m,org_diversity_1000m
0,54.965929,36.668509,Москва,0,0,0,0,0,0,0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,54.965929,36.676464,Москва,0,0,0,0,0,0,0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,54.965929,36.684419,Москва,0,0,0,0,0,0,0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,54.965929,36.692373,Москва,0,0,0,0,0,0,0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,54.965929,36.700328,Москва,0,0,0,0,0,0,0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


**Базовый прогон**

In [4]:
df = pd.read_csv('grid_features.csv')
df['target'] = (df['atm_count'] > 0).astype(int)

# Убираем банкоматные счётчики из фичей (утечка) + lat/lon + target
drop_cols = ['atm_count', 'sber_count', 'tinkoff_count', 'vtb_count',
             'alfa_count', 'gazprom_count', 'raiff_count',
             'lat', 'lon', 'target']
feature_cols = [c for c in df.columns if c not in drop_cols and c != 'city']

X = df[feature_cols].copy()
y = df['target'].values
groups = df['city'].values

# NaN в avg_levels - заполняем 0 (нет жилых домов)
X = X.fillna(0)

logo = LeaveOneGroupOut()
results = []

for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups)):
    city = groups[test_idx[0]]
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    model = CatBoostClassifier(
        iterations=500, depth=6, learning_rate=0.05,
        auto_class_weights='Balanced',
        eval_metric='AUC', verbose=100,
        random_seed=42
    )
    model.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)

    proba = model.predict_proba(X_test)[:, 1]
    roc = roc_auc_score(y_test, proba)
    pr = average_precision_score(y_test, proba)
    baseline = y_test.mean()

    results.append({'city': city, 'ROC-AUC': roc, 'PR-AUC': pr,
                    'baseline': baseline, 'PR/baseline': pr / baseline if baseline > 0 else 0})
    print(f"{city}: ROC-AUC={roc:.4f}, PR-AUC={pr:.4f}, PR/baseline={pr/baseline:.1f}x")

res = pd.DataFrame(results)
print("\nСРЕДНИЕ")
print(f"ROC-AUC:     {res['ROC-AUC'].mean():.4f}")
print(f"PR-AUC:      {res['PR-AUC'].mean():.4f}")
print(f"PR/baseline: {res['PR/baseline'].mean():.1f}x")

# Feature importance top-20
print("\nFEATURE IMPORTANCE (last fold)")
fi = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(fi.head(20).to_string())

0:	test: 0.9719680	best: 0.9719680 (0)	total: 238ms	remaining: 1m 58s
100:	test: 0.9886336	best: 0.9886336 (100)	total: 20.8s	remaining: 1m 22s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.9887925335
bestIteration = 130

Shrink model to first 131 iterations.
Казань: ROC-AUC=0.9888, PR-AUC=0.7628, PR/baseline=32.2x
0:	test: 0.9541021	best: 0.9541021 (0)	total: 162ms	remaining: 1m 20s
100:	test: 0.9845831	best: 0.9846080 (92)	total: 18.1s	remaining: 1m 11s
200:	test: 0.9852837	best: 0.9853176 (198)	total: 34.8s	remaining: 51.8s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.9854969164
bestIteration = 230

Shrink model to first 231 iterations.
Москва: ROC-AUC=0.9855, PR-AUC=0.7619, PR/baseline=19.6x
0:	test: 0.9842520	best: 0.9842520 (0)	total: 181ms	remaining: 1m 30s
100:	test: 0.9957733	best: 0.9957733 (100)	total: 17.7s	remaining: 1m 9s
200:	test: 0.9957828	best: 0.9958828 (166)	total: 36.3s	remaining: 54s
Stopped by overfitting detector  (50

Вообще такое различие roc auc от pr auc из-за дисбаланса классов. Вообще добавил pr_auc, т.к. у него нигде в знаменателе нет true negatives (TN). TN в roc_auc из-за большого количества негативов в моей выборке будет занижать FPR и искуственно "тянуть" график roc auc в левый верхний угол, выдавая искуственно хороший результат

**Умные негативы**

Попробуем более умную стратегию для негативов
0% - случайные точки, 30% - точки возле инфраструктуры, но без банкоматов. Остальное - соседние ячейки рядом с банкоматами

In [5]:
df = pd.read_csv('grid_features.csv')
df['target'] = (df['atm_count'] > 0).astype(int)

# Сборка выборки с умными негативами
positives = df[df['target'] == 1]
negatives = df[df['target'] == 0]

n_pos = len(positives)
# Негативов берём столько же, сколько позитивов (1:1)
n_neg = n_pos
n_random = int(n_neg * 0.10)
n_infra = int(n_neg * 0.30)
n_neighbors = n_neg - n_random - n_infra

print(f"Позитивы: {n_pos}")
print(f"Негативы: {n_neg} (random={n_random}, infra={n_infra}, neighbors={n_neighbors})")

np.random.seed(42)

# 1. случайные точки
random_neg = negatives.sample(n=n_random, random_state=42)

# 2. точки возле инфраструктуры (есть POI, но нет банкоматов)
infra_mask = (negatives['total_poi_500m'] > 3) & (negatives['orgs_500m'] > 5)
infra_candidates = negatives[infra_mask]
print(f"Кандидатов infra: {len(infra_candidates)}")
infra_neg = infra_candidates.sample(n=min(n_infra, len(infra_candidates)), random_state=42)

# 3. соседние ячейки (ближайшие по координатам к позитивам, без банкоматов)
pos_coords = positives[['lat', 'lon']].values
neg_coords = negatives[['lat', 'lon']].values

# для каждого позитива находим ближайшие негативы
tree = cKDTree(neg_coords)

k_per_pos = max(1, n_neighbors // n_pos + 1)
_, indices = tree.query(pos_coords, k=k_per_pos)
neighbor_indices = np.unique(indices.ravel())
np.random.shuffle(neighbor_indices)
neighbor_indices = neighbor_indices[:n_neighbors]
neighbor_neg = negatives.iloc[neighbor_indices]

# собираем финальный датасет
# убираем дубликаты по индексу
all_neg_idx = set(random_neg.index) | set(infra_neg.index) | set(neighbor_neg.index)
sampled_negatives = negatives.loc[list(all_neg_idx)]

df_sampled = pd.concat([positives, sampled_negatives]).sample(frac=1, random_state=42)
print(f"\nИтоговая выборка: {len(df_sampled)}, target rate: {df_sampled['target'].mean():.4f}")
print(df_sampled.groupby('city')['target'].agg(['mean', 'sum', 'count']))

Позитивы: 6264
Негативы: 6264 (random=626, infra=1879, neighbors=3759)
Кандидатов infra: 7982

Итоговая выборка: 11892, target rate: 0.5267
                     mean   sum  count
city                                  
Казань           0.507246   350    690
Москва           0.553313  3674   6640
Нижний Новгород  0.449126   437    973
Новосибирск      0.444444   520   1170
Санкт-Петербург  0.530384  1283   2419


In [6]:
# Убираем банкоматные счётчики из фичей (утечка) + lat/lon + target
drop_cols = ['atm_count', 'sber_count', 'tinkoff_count', 'vtb_count',
             'alfa_count', 'gazprom_count', 'raiff_count',
             'lat', 'lon', 'target']
feature_cols = [c for c in df_sampled.columns if c not in drop_cols and c != 'city']

X = df_sampled[feature_cols].copy()
y = df_sampled['target'].values
groups = df_sampled['city'].values

# NaN в avg_levels - заполняем 0 (нет жилых домов)
X = X.fillna(0)

logo = LeaveOneGroupOut()
results = []

for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups)):
    city = groups[test_idx[0]]
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    model = CatBoostClassifier(
        iterations=500, depth=6, learning_rate=0.05,
        auto_class_weights='Balanced',
        eval_metric='AUC', verbose=100,
        random_seed=42
    )
    model.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)

    proba = model.predict_proba(X_test)[:, 1]
    roc = roc_auc_score(y_test, proba)
    pr = average_precision_score(y_test, proba)
    baseline = y_test.mean()

    results.append({'city': city, 'ROC-AUC': roc, 'PR-AUC': pr,
                    'baseline': baseline, 'PR/baseline': pr / baseline if baseline > 0 else 0})
    print(f"{city}: ROC-AUC={roc:.4f}, PR-AUC={pr:.4f}, PR/baseline={pr/baseline:.1f}x")

res = pd.DataFrame(results)
print("\nСРЕДНИЕ")
print(f"ROC-AUC:     {res['ROC-AUC'].mean():.4f}")
print(f"PR-AUC:      {res['PR-AUC'].mean():.4f}")
print(f"PR/baseline: {res['PR/baseline'].mean():.1f}x")

# Feature importance top-20
print("\nFEATURE IMPORTANCE (last fold)")
fi = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(fi.head(20).to_string())

0:	test: 0.8219328	best: 0.8219328 (0)	total: 19.8ms	remaining: 9.88s
100:	test: 0.8639832	best: 0.8639832 (100)	total: 1.7s	remaining: 6.7s
200:	test: 0.8683361	best: 0.8683361 (200)	total: 3.53s	remaining: 5.25s
300:	test: 0.8686387	best: 0.8699748 (258)	total: 6.43s	remaining: 4.25s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.8699747899
bestIteration = 258

Shrink model to first 259 iterations.
Казань: ROC-AUC=0.8700, PR-AUC=0.8665, PR/baseline=1.7x
0:	test: 0.8079064	best: 0.8079064 (0)	total: 15.6ms	remaining: 7.77s
100:	test: 0.8594705	best: 0.8594705 (100)	total: 1.41s	remaining: 5.58s
200:	test: 0.8609968	best: 0.8615304 (192)	total: 2.83s	remaining: 4.22s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.8615303874
bestIteration = 192

Shrink model to first 193 iterations.
Москва: ROC-AUC=0.8615, PR-AUC=0.8840, PR/baseline=1.6x
0:	test: 0.8207845	best: 0.8207845 (0)	total: 18.8ms	remaining: 9.4s
100:	test: 0.8658253	best: 0.8658253 (10

Подберем гиперпараметры

In [ ]:
!pip install optuna
import optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 7.3 MB/s eta 0:00:00


In [ ]:
def objective(trial):
    params = {
        'iterations': 1000,
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 50),
        'random_strength': trial.suggest_float('random_strength', 0.1, 10, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0, 5),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'eval_metric': 'AUC',
        'verbose': 0,
        'random_seed': 42,
        'early_stopping_rounds': 50
    }

    logo = LeaveOneGroupOut()
    scores = []

    for train_idx, test_idx in logo.split(X, y, groups):
        model = CatBoostClassifier(**params)
        model.fit(X.iloc[train_idx], y[train_idx],
                  eval_set=(X.iloc[test_idx], y[test_idx]))

        proba = model.predict_proba(X.iloc[test_idx])[:, 1]
        scores.append(average_precision_score(y[test_idx], proba))

    return np.mean(scores)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print("\n=== ЛУЧШИЕ ПАРАМЕТРЫ ===")
print(study.best_params)
print(f"PR-AUC: {study.best_value:.4f}")

[I 2026-04-10 21:30:04,787] A new study created in memory with name: no-name-340275b0-db55-45b0-a684-d75043dff13b


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-04-10 21:30:14,786] Trial 0 finished with value: 0.8731056500113651 and parameters: {'depth': 5, 'learning_rate': 0.0714827144467258, 'l2_leaf_reg': 5.455605306230522, 'min_data_in_leaf': 17, 'random_strength': 0.21222129571331447, 'bagging_temperature': 2.7390939043996108, 'border_count': 90}. Best is trial 0 with value: 0.8731056500113651.
[I 2026-04-10 21:35:13,836] Trial 1 finished with value: 0.8678087193761215 and parameters: {'depth': 10, 'learning_rate': 0.018265763225709587, 'l2_leaf_reg': 6.48177576336432, 'min_data_in_leaf': 26, 'random_strength': 8.788495494583424, 'bagging_temperature': 0.28986337529424866, 'border_count': 121}. Best is trial 0 with value: 0.8731056500113651.
[I 2026-04-10 21:35:23,814] Trial 2 finished with value: 0.8665964767528218 and parameters: {'depth': 7, 'learning_rate': 0.0681935428657574, 'l2_leaf_reg': 2.376576713279265, 'min_data_in_leaf': 27, 'random_strength': 0.16955433402902614, 'bagging_temperature': 4.193005799217012, 'border_coun

In [7]:
best_params = {
    'iterations': 1000,
    'depth': 5,
    'learning_rate': 0.0715,
    'l2_leaf_reg': 5.456,
    'min_data_in_leaf': 17,
    'random_strength': 0.212,
    'bagging_temperature': 2.739,
    'border_count': 90,
    'eval_metric': 'AUC',
    'verbose': 100,
    'random_seed': 42,
    'early_stopping_rounds': 50
}

logo = LeaveOneGroupOut()
results = []

for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups)):
    city = groups[test_idx[0]]
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    model = CatBoostClassifier(**best_params
    )
    model.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)

    proba = model.predict_proba(X_test)[:, 1]
    roc = roc_auc_score(y_test, proba)
    pr = average_precision_score(y_test, proba)
    baseline = y_test.mean()

    results.append({'city': city, 'ROC-AUC': roc, 'PR-AUC': pr,
                    'baseline': baseline, 'PR/baseline': pr / baseline if baseline > 0 else 0})
    print(f"{city}: ROC-AUC={roc:.4f}, PR-AUC={pr:.4f}, PR/baseline={pr/baseline:.1f}x")

res = pd.DataFrame(results)
print("\nСРЕДНИЕ")
print(f"ROC-AUC:     {res['ROC-AUC'].mean():.4f}")
print(f"PR-AUC:      {res['PR-AUC'].mean():.4f}")
print(f"PR/baseline: {res['PR/baseline'].mean():.1f}x")

# Feature importance top-20
print("\nFEATURE IMPORTANCE (last fold)")
fi = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(fi.head(20).to_string())

0:	test: 0.8330546	best: 0.8330546 (0)	total: 9.46ms	remaining: 9.46s
100:	test: 0.8684034	best: 0.8686555 (96)	total: 868ms	remaining: 7.72s
200:	test: 0.8693697	best: 0.8696723 (165)	total: 1.69s	remaining: 6.72s
300:	test: 0.8702101	best: 0.8709664 (270)	total: 2.53s	remaining: 5.87s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.8709663866
bestIteration = 270

Shrink model to first 271 iterations.
Казань: ROC-AUC=0.8710, PR-AUC=0.8688, PR/baseline=1.7x
0:	test: 0.8262121	best: 0.8262121 (0)	total: 6.7ms	remaining: 6.69s
100:	test: 0.8616839	best: 0.8618393 (95)	total: 648ms	remaining: 5.77s
200:	test: 0.8620924	best: 0.8627291 (160)	total: 1.25s	remaining: 4.97s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.8627291026
bestIteration = 160

Shrink model to first 161 iterations.
Москва: ROC-AUC=0.8627, PR-AUC=0.8854, PR/baseline=1.6x
0:	test: 0.8394775	best: 0.8394775 (0)	total: 15.8ms	remaining: 15.7s
100:	test: 0.8727907	best: 0.8727907 (10

Подбор гиперпараметров через Optuna прироста почти не дал. Это говорит о том, что текущая модель уже достигла потолка на имеющихся фичах, и дальше тюнинг не помогает - нужно либо принципиально другие признаки, либо принципиально другая модель.

## **Начало Чекпоинта 6**

Попробуем метод представления ячеек, основанный на идеях hex2vec и tile2vec: будем deep MLP encoder обучать через contrastive InfoNCE loss так, чтобы пространственно близкие ячейки имели похожие эмбеддинги. В отличие от канонического hex2vec, который работает с категориальными OSM-тегами как с токенами, адаптация применяет contrastive learning напрямую к агрегированным числовым признакам ячеек

Дальше эмбеддинги будем использовать как доп. фичи для CatBoost

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import BallTree

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Device: cuda


In [10]:
df = pd.read_csv('grid_features.csv')
df['target'] = (df['atm_count'] > 0).astype(int)
print(f'Позитивов: {df["target"].sum()} ({df["target"].mean():.1%})')

# колонки с банковской информацией выкидываем, чтобы не было утечки
leak_cols = ['atm_count', 'sber_count', 'tinkoff_count', 'vtb_count',
             'alfa_count', 'gazprom_count', 'raiff_count']
exclude_cols = leak_cols + ['lat', 'lon', 'target', 'city']
feature_cols = [c for c in df.columns if c not in exclude_cols
                and pd.api.types.is_numeric_dtype(df[c])]
print(f'Признаков для encoder: {len(feature_cols)}')

Позитивов: 6264 (1.3%)
Признаков для encoder: 90


In [11]:
# для каждой ячейки находим её K ближайших соседей по координатам через BallTree с метрикой haversine
# эти соседи будут positives при contrastive обучении
coords_rad = np.radians(df[['lat', 'lon']].values)
tree = BallTree(coords_rad, metric='haversine')

K = 9  # себя + 8 соседей
_, neighbor_idx = tree.query(coords_rad, k=K)
neighbor_idx = neighbor_idx[:, 1:]  # убираем саму ячейку
print(f'Соседей на ячейку: {neighbor_idx.shape[1]}')

# нормализуем фичи
scaler = StandardScaler()
X_all = scaler.fit_transform(df[feature_cols].fillna(0).values).astype(np.float32)
print(f'Матрица признаков: {X_all.shape}')

Соседей на ячейку: 8
Матрица признаков: (496447, 90)


In [12]:
# датасет пар (anchor, positive_neighbor)
# anchor - сама ячейка, positive - случайный сосед
class NeighborPairDataset(Dataset):
    def __init__(self, X, neighbor_idx):
        self.X = X
        self.neighbor_idx = neighbor_idx
        self.n = len(X)

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        anchor = self.X[idx]
        pos_idx = np.random.choice(self.neighbor_idx[idx])
        positive = self.X[pos_idx]
        return torch.from_numpy(anchor), torch.from_numpy(positive)


dataset = NeighborPairDataset(X_all, neighbor_idx)
loader = DataLoader(dataset, batch_size=512, shuffle=True, num_workers=2, drop_last=True)

In [13]:
# deep MLP encoder: 4 полносвязных блока с BatchNorm + ReLU + Dropout
# на выходе - 64-мерный нормализованный эмбеддинг
class DeepEncoder(nn.Module):
    def __init__(self, in_dim, hidden=256, embed_dim=64, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden, hidden // 2),
            nn.BatchNorm1d(hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden // 2, embed_dim),
        )

    def forward(self, x):
        z = self.net(x)
        # L2-нормализация - стандарт для contrastive learning
        return F.normalize(z, dim=-1)


# InfoNCE loss из SimCLR: для anchor positive должен быть ближе всех negatives в батче
# negatives берутся из других элементов батча
def info_nce_loss(z_anchor, z_pos, temperature=0.1):
    logits = z_anchor @ z_pos.T / temperature
    labels = torch.arange(len(z_anchor), device=z_anchor.device)
    return F.cross_entropy(logits, labels)


encoder = DeepEncoder(in_dim=len(feature_cols)).to(device)
optimizer = torch.optim.AdamW(encoder.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

print(f'Параметров в encoder: {sum(p.numel() for p in encoder.parameters()):,}')

Параметров в encoder: 131,520


In [14]:
# обучение энкодера
EPOCHS = 10
encoder.train()

for epoch in range(EPOCHS):
    total_loss = 0.0
    n_batches = 0

    for anchor, positive in loader:
        anchor = anchor.to(device)
        positive = positive.to(device)

        z_a = encoder(anchor)
        z_p = encoder(positive)

        loss = info_nce_loss(z_a, z_p)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    scheduler.step()
    avg_loss = total_loss / n_batches
    print(f'Epoch {epoch+1:2d}/{EPOCHS} | loss={avg_loss:.4f} | lr={scheduler.get_last_lr()[0]:.2e}')

Epoch  1/10 | loss=1.4116 | lr=9.76e-04
Epoch  2/10 | loss=1.0079 | lr=9.05e-04
Epoch  3/10 | loss=0.9231 | lr=7.94e-04
Epoch  4/10 | loss=0.8784 | lr=6.55e-04
Epoch  5/10 | loss=0.8515 | lr=5.00e-04
Epoch  6/10 | loss=0.8270 | lr=3.45e-04
Epoch  7/10 | loss=0.8103 | lr=2.06e-04
Epoch  8/10 | loss=0.7953 | lr=9.55e-05
Epoch  9/10 | loss=0.7874 | lr=2.45e-05
Epoch 10/10 | loss=0.7821 | lr=0.00e+00


In [15]:
# извлекаем эмбеддинги для всех ячеек батчами по 4096
encoder.eval()
embeddings = []
BATCH = 4096

with torch.no_grad():
    for i in range(0, len(X_all), BATCH):
        batch = torch.from_numpy(X_all[i:i+BATCH]).to(device)
        emb = encoder(batch).cpu().numpy()
        embeddings.append(emb)

embeddings = np.vstack(embeddings)
print(f'Эмбеддинги: {embeddings.shape}')

# добавляем в df как новые фичи emb_0 ... emb_63
embed_cols = [f'emb_{i}' for i in range(embeddings.shape[1])]
df_emb = pd.DataFrame(embeddings, columns=embed_cols, index=df.index)
df = pd.concat([df, df_emb], axis=1)

Эмбеддинги: (496447, 64)


Соберём сбалансированную выборку с умными негативами по той же схеме, что в бейзлайне (10/30/60)

In [16]:
positives = df[df['target'] == 1]
negatives = df[df['target'] == 0]

n_pos = len(positives)
n_neg = n_pos
n_random = int(n_neg * 0.10)
n_infra = int(n_neg * 0.30)
n_neighbors = n_neg - n_random - n_infra

print(f'Позитивы: {n_pos}')
print(f'Негативы: {n_neg} (random={n_random}, infra={n_infra}, neighbors={n_neighbors})')

np.random.seed(SEED)

# 1. случайные точки
random_neg = negatives.sample(n=n_random, random_state=SEED)

# 2. точки возле инфраструктуры
infra_mask = (negatives['total_poi_500m'] > 3) & (negatives['orgs_500m'] > 5)
infra_candidates = negatives[infra_mask]
print(f'Кандидатов infra: {len(infra_candidates)}')
infra_neg = infra_candidates.sample(n=min(n_infra, len(infra_candidates)), random_state=SEED)

# 3. соседние ячейки (ближайшие по координатам к позитивам)
pos_coords = positives[['lat', 'lon']].values
neg_coords = negatives[['lat', 'lon']].values

tree_neg = cKDTree(neg_coords)
k_per_pos = max(1, n_neighbors // n_pos + 1)
_, indices = tree_neg.query(pos_coords, k=k_per_pos)
neighbor_indices = np.unique(indices.ravel())
np.random.shuffle(neighbor_indices)
neighbor_indices = neighbor_indices[:n_neighbors]
neighbor_neg = negatives.iloc[neighbor_indices]

# финальный датасет - убираем дубликаты по индексу
all_neg_idx = set(random_neg.index) | set(infra_neg.index) | set(neighbor_neg.index)
sampled_negatives = negatives.loc[list(all_neg_idx)]

df_sampled = pd.concat([positives, sampled_negatives]).sample(frac=1, random_state=SEED)
print(f'\nИтоговая выборка: {len(df_sampled)}, target rate: {df_sampled["target"].mean():.4f}')
print(df_sampled.groupby('city')['target'].agg(['mean', 'sum', 'count']))

Позитивы: 6264
Негативы: 6264 (random=626, infra=1879, neighbors=3759)
Кандидатов infra: 7982

Итоговая выборка: 11892, target rate: 0.5267
                     mean   sum  count
city                                  
Казань           0.507246   350    690
Москва           0.553313  3674   6640
Нижний Новгород  0.449126   437    973
Новосибирск      0.444444   520   1170
Санкт-Петербург  0.530384  1283   2419


In [17]:
# для дальнейших сравнений результатов моделей создадим бенчмарк словарь
benchmark = {}

In [18]:
tabular_cols = [c for c in feature_cols]
combined_cols = tabular_cols + embed_cols # 95 + 64 = 159 фичей


# универсальная функция для оценки CatBoost на разных наборах фичей
# возвращает датафрейм с метриками и последнюю обученную модель
def evaluate(feature_set, name):
    X = df_sampled[feature_set].fillna(0).copy()
    y = df_sampled['target'].values
    groups = df_sampled['city'].values

    logo = LeaveOneGroupOut()
    results = []
    last_model = None

    for fold, (tr_idx, te_idx) in enumerate(logo.split(X, y, groups)):
        city = groups[te_idx[0]]
        X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
        y_tr, y_te = y[tr_idx], y[te_idx]

        model = CatBoostClassifier(
            iterations=500,
            depth=6,
            learning_rate=0.05,
            auto_class_weights='Balanced',
            eval_metric='AUC',
            verbose=0,
            random_seed=SEED
        )
        model.fit(X_tr, y_tr, eval_set=(X_te, y_te), early_stopping_rounds=50)
        last_model = model

        proba = model.predict_proba(X_te)[:, 1]
        roc = roc_auc_score(y_te, proba)
        pr = average_precision_score(y_te, proba)
        baseline = y_te.mean()

        results.append({
            'city': city,
            'ROC-AUC': roc,
            'PR-AUC': pr,
            'baseline': baseline,
            'PR/baseline': pr / baseline if baseline > 0 else 0
        })
        print(f'  [{name}] {city}: ROC={roc:.4f} PR={pr:.4f} PR/base={pr/baseline:.2f}x')

    res = pd.DataFrame(results)
    print(f'  [{name}] СРЕДНЕЕ: ROC={res["ROC-AUC"].mean():.4f} '
          f'PR={res["PR-AUC"].mean():.4f} PR/base={res["PR/baseline"].mean():.2f}x')
    return res, last_model


print('=' * 70)
print('БЕЗ ЭМБЕДДИНГОВ (CatBoost-baseline на 95 фичах):')
print('=' * 70)
res_base, _ = evaluate(tabular_cols, 'baseline')

print()
print('=' * 70)
print('С ЭМБЕДДИНГАМИ (95 + 64 = 159 фичей):')
print('=' * 70)
res_emb, model_emb = evaluate(combined_cols, 'with_emb')

print()
print('=' * 70)
print('СРАВНЕНИЕ:')
print('=' * 70)
print(f'CatBoost-baseline:    ROC={res_base["ROC-AUC"].mean():.4f}  '
      f'PR={res_base["PR-AUC"].mean():.4f}  '
      f'PR/base={res_base["PR/baseline"].mean():.2f}x')
print(f'CatBoost + эмбеддинги: ROC={res_emb["ROC-AUC"].mean():.4f}  '
      f'PR={res_emb["PR-AUC"].mean():.4f}  '
      f'PR/base={res_emb["PR/baseline"].mean():.2f}x')
print(f'Дельта PR-AUC: {res_emb["PR-AUC"].mean() - res_base["PR-AUC"].mean():+.4f}')

# добавляем в бенчмарк для сравнения моделей
benchmark['CatBoost_baseline'] = [
    res_base['ROC-AUC'].mean(),
    res_base['PR-AUC'].mean(),
    res_base['PR/baseline'].mean()
]
benchmark['CatBoost_+_contrastive_embeddings'] = [
    res_emb['ROC-AUC'].mean(),
    res_emb['PR-AUC'].mean(),
    res_emb['PR/baseline'].mean()
]

БЕЗ ЭМБЕДДИНГОВ (CatBoost-baseline на 95 фичах):
  [baseline] Казань: ROC=0.8700 PR=0.8665 PR/base=1.71x
  [baseline] Москва: ROC=0.8615 PR=0.8840 PR/base=1.60x
  [baseline] Нижний Новгород: ROC=0.8742 PR=0.8525 PR/base=1.90x
  [baseline] Новосибирск: ROC=0.9050 PR=0.8859 PR/base=1.99x
  [baseline] Санкт-Петербург: ROC=0.8558 PR=0.8600 PR/base=1.62x
  [baseline] СРЕДНЕЕ: ROC=0.8733 PR=0.8698 PR/base=1.76x

С ЭМБЕДДИНГАМИ (95 + 64 = 159 фичей):
  [with_emb] Казань: ROC=0.8698 PR=0.8667 PR/base=1.71x
  [with_emb] Москва: ROC=0.8609 PR=0.8841 PR/base=1.60x
  [with_emb] Нижний Новгород: ROC=0.8705 PR=0.8517 PR/base=1.90x
  [with_emb] Новосибирск: ROC=0.8996 PR=0.8784 PR/base=1.98x
  [with_emb] Санкт-Петербург: ROC=0.8555 PR=0.8571 PR/base=1.62x
  [with_emb] СРЕДНЕЕ: ROC=0.8713 PR=0.8676 PR/base=1.76x

СРАВНЕНИЕ:
CatBoost-baseline:    ROC=0.8733  PR=0.8698  PR/base=1.76x
CatBoost + эмбеддинги: ROC=0.8713  PR=0.8676  PR/base=1.76x
Дельта PR-AUC: -0.0022


In [19]:
# смотрим, попали ли эмбеддинги в топ важных фичей
imp = pd.Series(model_emb.feature_importances_, index=combined_cols).sort_values(ascending=False)
print('Топ-25 фичей (модель с эмбеддингами):')
print(imp.head(25).to_string())

n_emb_in_top25 = sum(1 for f in imp.head(25).index if f.startswith('emb_'))
print(f'\nЭмбеддингов в топ-25: {n_emb_in_top25}/64')
print(f'Суммарная важность всех эмбеддингов: {imp[embed_cols].sum():.2f}')
print(f'Суммарная важность табличных фичей: {imp[tabular_cols].sum():.2f}')

Топ-25 фичей (модель с эмбеддингами):
nearest_pharmacies          11.340897
nearest_malls                8.141229
nearest_pyaterochka          6.046964
nearest_bus_stops            3.678122
nearest_cafes                3.416903
orgs_500m                    2.437157
nearest_post_offices         2.308446
nearest_magnit               2.264645
nearest_perekrestok          2.172538
nearest_gas_stations         1.930649
nearest_mfc                  1.893682
nearest_business_centres     1.863000
nearest_lenta                1.785339
nearest_fix_price            1.627767
nearest_metro                1.454568
nearest_hospitals            1.255113
nearest_diksi                1.175847
residential_ratio_500m       1.162336
nearest_auchan               1.160007
org_diversity_500m           1.090417
nearest_residential          1.005840
nearest_parks                1.004846
malls_1000m                  0.874851
poi_diversity_500m           0.852382
nearest_markets              0.829030

Эмбеддингов

Эмбеддинги в топ-25 не попали ни разу, хотя суммарная важность всех 64 эмбеддингов составила около 22%. Это говорит о том, что CatBoost их использует, но как слабые шумовые сигналы. Прироста PR-AUC нет, потому что encoder обучался на тех же 95 признаках, что доступны CatBoost, и не привнёс новой информации - пространственный контекст, закодированный через близость соседей в эмбеддинг-пространстве, уже представлен в исходных признаках через радиусы 500м/1000м
Для прорыва нужны методы, которые явно используют признаки соседних ячеек, а не пытаются переупаковать признаки самой ячейки Поэтому плавно переходим к графовой нейросети

**GraphSAGE**

Идея: строим граф, где узлы - ячейки, рёбра - пространственное соседство. GraphSAGE на каждом слое агрегирует признаки соседей. Ключевое отличие от предыдущих экспериментов: GNN получает признаки соседних ячеек как новый источник информации, а не переупаковывает признаки одной ячейки. Это решает проблему, диагностированную в эксперименте со spatial contrastive embeddings

Используем модель в двух режимах:
1. End-to-end: GraphSAGE -> классификация напрямую
2. Эмбеддинги для CatBoost: GraphSAGE учит представления, дальше CatBoost на (95 табличных + 64 GNN-эмбеддинга)

Валидация LOGO по городам - граф один, но loss и оценка на разных подмножествах узлов

In [4]:
import torch
TORCH_VER = torch.__version__.split('+')[0]
CUDA_VER = 'cu121' if torch.cuda.is_available() else 'cpu'
print(f'torch: {TORCH_VER}, cuda: {CUDA_VER}')

!pip install -q torch_geometric

torch: 2.10.0, cuda: cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 36.1 MB/s eta 0:00:00


In [5]:
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Device: cuda


In [6]:
df = pd.read_csv('grid_features.csv').reset_index(drop=True)
print(f'Всего ячеек: {len(df)}')

df['target'] = (df['atm_count'] > 0).astype(int)
print(f'Позитивов: {df["target"].sum()} ({df["target"].mean():.1%})')

leak_cols = ['atm_count', 'sber_count', 'tinkoff_count', 'vtb_count',
             'alfa_count', 'gazprom_count', 'raiff_count']
exclude_cols = leak_cols + ['lat', 'lon', 'target', 'city']
feature_cols = [c for c in df.columns if c not in exclude_cols
                and pd.api.types.is_numeric_dtype(df[c])]
print(f'Признаков узла: {len(feature_cols)}')

Всего ячеек: 496447
Позитивов: 6264 (1.3%)
Признаков узла: 90


In [9]:
# признаки узлов
scaler = StandardScaler()
X = scaler.fit_transform(df[feature_cols].fillna(0).values).astype(np.float32)
y = df['target'].values.astype(np.int64)

# граф: 8 ближайших соседей по координатам (метрика haversine)
K = 8
coords_rad = np.radians(df[['lat', 'lon']].values)
tree = BallTree(coords_rad, metric='haversine')
_, neighbor_idx = tree.query(coords_rad, k=K + 1) # +1 потому что включает саму ячейку
neighbor_idx = neighbor_idx[:, 1:] # убираем саму ячейку

# edge_index в формате PyG: [2, num_edges]
src = np.repeat(np.arange(len(df)), K)
dst = neighbor_idx.flatten()
edge_index = np.stack([src, dst], axis=0)

# делаем граф неориентированным (добавляем обратные рёбра) и убираем дубликаты
edge_index = np.concatenate([edge_index, edge_index[[1, 0]]], axis=1)
edge_index = np.unique(edge_index, axis=1)

print(f'Узлов: {len(df)}')
print(f'Рёбер: {edge_index.shape[1]:,}')
print(f'Среднее число соседей: {edge_index.shape[1] / len(df):.1f}')

# заворачиваем в PyG Data
data = Data(
    x=torch.from_numpy(X),
    edge_index=torch.from_numpy(edge_index).long(),
    y=torch.from_numpy(y),
)
print(data)

Узлов: 496447
Рёбер: 3,977,700
Среднее число соседей: 8.0
Data(x=[496447, 90], edge_index=[2, 3977700], y=[496447])


In [10]:
# функция для получения индексов сбалансированной выборки
# нужна для GNN, чтобы train-маски строились по той же логике 10/30/60, что и в CatBoost бейзлайне
def get_smart_sample_indices(df, exclude_city=None):
    """Возвращает индексы (в df) выборки с умными негативами.
    Если exclude_city задан - исключает этот город из выборки (для LOGO)."""
    if exclude_city is not None:
        mask = df['city'] != exclude_city
        sub = df[mask]
    else:
        sub = df

    positives = sub[sub['target'] == 1]
    negatives = sub[sub['target'] == 0]

    n_pos = len(positives)
    n_neg = n_pos
    n_random = int(n_neg * 0.10)
    n_infra = int(n_neg * 0.30)
    n_neighbors = n_neg - n_random - n_infra

    rng = np.random.RandomState(SEED)

    random_neg = negatives.sample(n=n_random, random_state=SEED)

    infra_mask = (negatives['total_poi_500m'] > 3) & (negatives['orgs_500m'] > 5)
    infra_candidates = negatives[infra_mask]
    infra_neg = infra_candidates.sample(
        n=min(n_infra, len(infra_candidates)),
        random_state=SEED
    )

    pos_coords = positives[['lat', 'lon']].values
    neg_coords = negatives[['lat', 'lon']].values
    tree_neg = cKDTree(neg_coords)
    k_per_pos = max(1, n_neighbors // n_pos + 1)
    _, indices = tree_neg.query(pos_coords, k=k_per_pos)
    neighbor_indices = np.unique(indices.ravel())
    rng.shuffle(neighbor_indices)
    neighbor_indices = neighbor_indices[:n_neighbors]
    neighbor_neg = negatives.iloc[neighbor_indices]

    all_neg_idx = set(random_neg.index) | set(infra_neg.index) | set(neighbor_neg.index)
    sampled_idx = list(set(positives.index) | all_neg_idx)
    return np.array(sorted(sampled_idx))

In [11]:
# архитектура GraphSAGE: 3 слоя SAGEConv с агрегатором mean
# после каждого слоя - BatchNorm + ReLU + Dropout
# две "головы": embed_head для эмбеддингов, classifier для классификации напрямую
class GraphSAGE(nn.Module):
    def __init__(self, in_dim, hidden=256, embed_dim=64, dropout=0.3):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hidden, aggr='mean')
        self.bn1 = nn.BatchNorm1d(hidden)

        self.conv2 = SAGEConv(hidden, hidden, aggr='mean')
        self.bn2 = nn.BatchNorm1d(hidden)

        self.conv3 = SAGEConv(hidden, hidden, aggr='mean')
        self.bn3 = nn.BatchNorm1d(hidden)

        # голова для эмбеддингов
        self.embed_head = nn.Linear(hidden, embed_dim)

        # голова для классификации
        self.classifier = nn.Linear(hidden, 1)

        self.dropout = dropout

    def forward(self, x, edge_index):
        h = self.conv1(x, edge_index)
        h = self.bn1(h)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)

        h = self.conv2(h, edge_index)
        h = self.bn2(h)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)

        h = self.conv3(h, edge_index)
        h = self.bn3(h)
        h = F.relu(h)
        h_final = F.dropout(h, p=self.dropout, training=self.training)

        # логиты классификации
        logits = self.classifier(h_final).squeeze(-1)
        # эмбеддинги (без dropout, нормализованные)
        embeddings = F.normalize(self.embed_head(h), dim=-1)

        return logits, embeddings


# тест архитектуры (просто чтобы посчитать число параметров)
model_test = GraphSAGE(in_dim=len(feature_cols)).to(device)
print(f'Параметров: {sum(p.numel() for p in model_test.parameters()):,}')
del model_test

Параметров: 327,233


In [12]:
EPOCHS = 200
LR = 1e-3
WD = 1e-5
PATIENCE = 25  # сколько эпох ждём улучшения PR-AUC до остановки

# граф на устройство (если уже на GPU - повторно не навредит)
data = data.to(device)

# сбалансированная выборка для оценки (та же что у CatBoost)
# здесь не передаём exclude_city - берём со всех городов, а LOGO режем по городам
all_sampled_idx = get_smart_sample_indices(df, exclude_city=None)
df_sampled = df.loc[all_sampled_idx]
print(f'Сбалансированная выборка: {len(df_sampled)} ячеек')
print(df_sampled.groupby('city')['target'].agg(['mean', 'sum', 'count']))

cities_unique = df['city'].unique()
print(f'\nГорода: {list(cities_unique)}')

Сбалансированная выборка: 11892 ячеек
                     mean   sum  count
city                                  
Казань           0.507246   350    690
Москва           0.553313  3674   6640
Нижний Новгород  0.449126   437    973
Новосибирск      0.444444   520   1170
Санкт-Петербург  0.530384  1283   2419

Города: ['Москва', 'Санкт-Петербург', 'Нижний Новгород', 'Новосибирск', 'Казань']


In [15]:
# обучение GraphSAGE на одном fold'е LOGO
# возвращает метрики и эмбеддинги всех узлов на лучшей эпохе
def train_one_fold(test_city, epochs=EPOCHS, patience=PATIENCE, verbose=False):
    """Обучает GraphSAGE с test_city как hold-out.
    Возвращает: метрики, эмбеддинги для всех узлов (для использования в CatBoost)."""

    train_idx = df_sampled[df_sampled['city'] != test_city].index.values
    test_idx = df_sampled[df_sampled['city'] == test_city].index.values

    train_mask = torch.zeros(len(df), dtype=torch.bool, device=device)
    train_mask[train_idx] = True
    test_mask = torch.zeros(len(df), dtype=torch.bool, device=device)
    test_mask[test_idx] = True

    # веса классов для balanced loss
    y_train = data.y[train_mask].cpu().numpy()
    pos_weight = (y_train == 0).sum() / max(1, (y_train == 1).sum())
    pos_weight_t = torch.tensor(pos_weight, dtype=torch.float, device=device)

    model = GraphSAGE(in_dim=len(feature_cols), hidden=128).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_pr = 0.0
    best_roc = 0.0
    best_embeddings = None
    epochs_without_improve = 0
    stopped_epoch = epochs

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        logits, _ = model(data.x, data.edge_index)
        loss = F.binary_cross_entropy_with_logits(
            logits[train_mask], data.y[train_mask].float(),
            pos_weight=pos_weight_t,
        )
        loss.backward()
        optimizer.step()
        scheduler.step()

        # оценка на тестовом городе
        model.eval()
        with torch.no_grad():
            logits, embeddings = model(data.x, data.edge_index)
            probs = torch.sigmoid(logits[test_mask]).cpu().numpy()
            y_test = data.y[test_mask].cpu().numpy()
            pr = average_precision_score(y_test, probs)
            roc = roc_auc_score(y_test, probs)

        if pr > best_pr:
            best_pr = pr
            best_roc = roc
            best_embeddings = embeddings.detach().cpu().numpy()
            epochs_without_improve = 0
        else:
            epochs_without_improve += 1

        if verbose and (epoch + 1) % 20 == 0:
            print(f'    Epoch {epoch+1:3d}: loss={loss.item():.4f} '
                  f'ROC={roc:.4f} PR={pr:.4f} (best PR={best_pr:.4f})')

        # early stopping
        if epochs_without_improve >= patience:
            stopped_epoch = epoch + 1
            if verbose:
                print(f'    Early stop на эпохе {stopped_epoch} '
                      f'(no improvement за {patience} эпох)')
            break

    baseline = data.y[test_mask].float().mean().item()
    return {
        'city': test_city,
        'ROC-AUC': best_roc,
        'PR-AUC': best_pr,
        'baseline': baseline,
        'PR/baseline': best_pr / baseline if baseline > 0 else 0,
        'stopped_epoch': stopped_epoch,
        'embeddings': best_embeddings,
    }

In [19]:
# запуск LOGO для GraphSAGE end-to-end
print('=' * 70)
print('GraphSAGE end-to-end (классификация напрямую)')
print('=' * 70)

gnn_results = []
fold_embeddings = {}  # эмбеддинги от каждого fold'а - для последующего CatBoost

for test_city in cities_unique:
    print(f'\nFold: hold-out = {test_city}')
    res = train_one_fold(test_city, epochs=EPOCHS, verbose=True)
    fold_embeddings[test_city] = res.pop('embeddings')
    gnn_results.append(res)
    print(f'  -> ROC={res["ROC-AUC"]:.4f} PR={res["PR-AUC"]:.4f} '
          f'PR/base={res["PR/baseline"]:.2f}x')

gnn_df = pd.DataFrame(gnn_results)
print('\n' + '=' * 70)
print('GraphSAGE end-to-end СРЕДНЕЕ:')
print(f'  ROC-AUC:     {gnn_df["ROC-AUC"].mean():.4f}')
print(f'  PR-AUC:      {gnn_df["PR-AUC"].mean():.4f}')
print(f'  PR/baseline: {gnn_df["PR/baseline"].mean():.2f}x')

# добавляем в бенчмарк для сравнения моделей
benchmark['GraphSAGE_end_to_end'] = [
    gnn_df['ROC-AUC'].mean(),
    gnn_df['PR-AUC'].mean(),
    gnn_df['PR/baseline'].mean()]

GraphSAGE end-to-end (классификация напрямую)

Fold: hold-out = Москва
    Epoch  20: loss=0.6340 ROC=0.7588 PR=0.7899 (best PR=0.7905)
    Epoch  40: loss=0.5752 ROC=0.7749 PR=0.8042 (best PR=0.8042)
    Epoch  60: loss=0.5513 ROC=0.7804 PR=0.8095 (best PR=0.8095)
    Epoch  80: loss=0.5385 ROC=0.7808 PR=0.8096 (best PR=0.8097)
    Epoch 100: loss=0.5286 ROC=0.7804 PR=0.8096 (best PR=0.8100)
    Early stop на эпохе 109 (no improvement за 25 эпох)
  -> ROC=0.7807 PR=0.8100 PR/base=1.46x

Fold: hold-out = Санкт-Петербург
    Epoch  20: loss=0.5994 ROC=0.7174 PR=0.7362 (best PR=0.7378)
    Epoch  40: loss=0.5487 ROC=0.7534 PR=0.7732 (best PR=0.7743)
    Epoch  60: loss=0.5301 ROC=0.7546 PR=0.7718 (best PR=0.7753)
    Early stop на эпохе 73 (no improvement за 25 эпох)
  -> ROC=0.7580 PR=0.7753 PR/base=1.46x

Fold: hold-out = Нижний Новгород
    Epoch  20: loss=0.5548 ROC=0.7252 PR=0.7119 (best PR=0.7131)
    Epoch  40: loss=0.5259 ROC=0.7529 PR=0.7360 (best PR=0.7362)
    Epoch  60: loss=

**CatBoost + GNN-эмбеддинги**

Стратегия из рекомендаций куратора: GraphSAGE учит представления узлов через message passing, дальше CatBoost обучается на (95 табличных + 64 GNN-эмбеддинга). Для каждого фолда используем именно те эмбеддинги, что выучились без участия тестового города

In [20]:
print('=' * 70)
print('CatBoost на (95 табличных + 64 GNN-эмбеддинга)')
print('=' * 70)

catboost_gnn_results = []

for test_city in cities_unique:
    embeddings = fold_embeddings[test_city]                       # [N, 64]
    embed_cols = [f'gnn_emb_{i}' for i in range(embeddings.shape[1])]

    # собираем фрейм: табличные фичи + GNN-эмбеддинги
    df_combined = df_sampled.copy()
    emb_for_sampled = embeddings[df_sampled.index.values]
    for i, col in enumerate(embed_cols):
        df_combined[col] = emb_for_sampled[:, i]

    feat_combined = feature_cols + embed_cols

    train_mask = df_combined['city'] != test_city
    X_tr = df_combined.loc[train_mask, feat_combined].fillna(0)
    y_tr = df_combined.loc[train_mask, 'target'].values
    X_te = df_combined.loc[~train_mask, feat_combined].fillna(0)
    y_te = df_combined.loc[~train_mask, 'target'].values

    model = CatBoostClassifier(
        iterations=500,
        depth=6,
        learning_rate=0.05,
        auto_class_weights='Balanced',
        eval_metric='AUC',
        verbose=0,
        random_seed=SEED
    )
    model.fit(X_tr, y_tr, eval_set=(X_te, y_te), early_stopping_rounds=50)

    proba = model.predict_proba(X_te)[:, 1]
    roc = roc_auc_score(y_te, proba)
    pr = average_precision_score(y_te, proba)
    baseline = y_te.mean()

    catboost_gnn_results.append({
        'city': test_city,
        'ROC-AUC': roc,
        'PR-AUC': pr,
        'baseline': baseline,
        'PR/baseline': pr / baseline if baseline > 0 else 0,
    })
    print(f'  {test_city}: ROC={roc:.4f} PR={pr:.4f} PR/base={pr/baseline:.2f}x')

cgnn_df = pd.DataFrame(catboost_gnn_results)
print('\n' + '=' * 70)
print('CatBoost + GNN-эмбеддинги СРЕДНЕЕ:')
print(f'  ROC-AUC:     {cgnn_df["ROC-AUC"].mean():.4f}')
print(f'  PR-AUC:      {cgnn_df["PR-AUC"].mean():.4f}')
print(f'  PR/baseline: {cgnn_df["PR/baseline"].mean():.2f}x')

# добавляем в бенчмарк для сравнения моделей
benchmark['CatBoost_+_GNN_embeddings'] = [
    cgnn_df['ROC-AUC'].mean(),
    cgnn_df['PR-AUC'].mean(),
    cgnn_df['PR/baseline'].mean()
]

CatBoost на (95 табличных + 64 GNN-эмбеддинга)
  Москва: ROC=0.8524 PR=0.8782 PR/base=1.59x
  Санкт-Петербург: ROC=0.8552 PR=0.8595 PR/base=1.62x
  Нижний Новгород: ROC=0.8703 PR=0.8546 PR/base=1.90x
  Новосибирск: ROC=0.9041 PR=0.8839 PR/base=1.99x
  Казань: ROC=0.8687 PR=0.8620 PR/base=1.70x

CatBoost + GNN-эмбеддинги СРЕДНЕЕ:
  ROC-AUC:     0.8701
  PR-AUC:      0.8676
  PR/baseline: 1.76x


**MLP-классификатор**

Простой DL-baseline для сравнения. Идея - показать, что нейросеть на табличных фичах без пространственной структуры работает хуже CatBoost. CatBoost > MLP > GraphSAGE-end-to-end.

In [21]:
from torch.utils.data import TensorDataset, DataLoader


# простой MLP-классификатор (3 скрытых блока: Linear + BN + ReLU + Dropout)
class MLPClassifier(nn.Module):
    def __init__(self, in_dim, hidden=256, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden, hidden // 2),
            nn.BatchNorm1d(hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden // 2, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


# обучение MLP на одном fold'е LOGO
def train_mlp_fold(test_city, epochs=100, batch_size=256, patience=15, verbose=False):
    """Обучает MLP на сбалансированной выборке с test_city как hold-out."""

    # train/test индексы из сбалансированной выборки
    train_df = df_sampled[df_sampled['city'] != test_city]
    test_df = df_sampled[df_sampled['city'] == test_city]

    # нормализуем фичи только по тренировочной части (чтобы не было утечки статистик с теста)
    scaler_local = StandardScaler()
    X_tr = scaler_local.fit_transform(train_df[feature_cols].fillna(0).values).astype(np.float32)
    X_te = scaler_local.transform(test_df[feature_cols].fillna(0).values).astype(np.float32)
    y_tr = train_df['target'].values.astype(np.float32)
    y_te = test_df['target'].values.astype(np.float32)

    X_tr_t = torch.from_numpy(X_tr).to(device)
    y_tr_t = torch.from_numpy(y_tr).to(device)
    X_te_t = torch.from_numpy(X_te).to(device)

    # DataLoader для батчей
    train_ds = TensorDataset(X_tr_t, y_tr_t)
    loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)

    # веса классов для balanced loss
    pos_weight = (y_tr == 0).sum() / max(1, (y_tr == 1).sum())
    pos_weight_t = torch.tensor(pos_weight, dtype=torch.float, device=device)

    model = MLPClassifier(in_dim=len(feature_cols)).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_pr = 0.0
    best_roc = 0.0
    epochs_without_improve = 0
    stopped_epoch = epochs

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        n_batches = 0
        for xb, yb in loader:
            logits = model(xb)
            loss = F.binary_cross_entropy_with_logits(logits, yb, pos_weight=pos_weight_t)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            n_batches += 1
        scheduler.step()

        # оценка
        model.eval()
        with torch.no_grad():
            probs = torch.sigmoid(model(X_te_t)).cpu().numpy()
        pr = average_precision_score(y_te, probs)
        roc = roc_auc_score(y_te, probs)

        if pr > best_pr:
            best_pr = pr
            best_roc = roc
            epochs_without_improve = 0
        else:
            epochs_without_improve += 1

        if verbose and (epoch + 1) % 20 == 0:
            print(f'    Epoch {epoch+1:3d}: loss={epoch_loss/n_batches:.4f} '
                  f'ROC={roc:.4f} PR={pr:.4f} (best PR={best_pr:.4f})')

        if epochs_without_improve >= patience:
            stopped_epoch = epoch + 1
            if verbose:
                print(f'    Early stop на эпохе {stopped_epoch}')
            break

    baseline = y_te.mean()
    return {
        'city': test_city,
        'ROC-AUC': best_roc,
        'PR-AUC': best_pr,
        'baseline': baseline,
        'PR/baseline': best_pr / baseline if baseline > 0 else 0,
        'stopped_epoch': stopped_epoch,
    }

In [23]:
# запуск LOGO для MLP
print('=' * 70)
print('MLP-классификатор (тот же DL, но без графовой структуры)')
print('=' * 70)

mlp_results = []
for test_city in cities_unique:
    print(f'\nFold: hold-out = {test_city}')
    res = train_mlp_fold(test_city, epochs=100, verbose=True)
    mlp_results.append(res)
    print(f'  -> ROC={res["ROC-AUC"]:.4f} PR={res["PR-AUC"]:.4f} '
          f'PR/base={res["PR/baseline"]:.2f}x (stopped at {res["stopped_epoch"]})')

mlp_df = pd.DataFrame(mlp_results)
print('\n' + '=' * 70)
print('MLP СРЕДНЕЕ:')
print(f'  ROC-AUC:     {mlp_df["ROC-AUC"].mean():.4f}')
print(f'  PR-AUC:      {mlp_df["PR-AUC"].mean():.4f}')
print(f'  PR/baseline: {mlp_df["PR/baseline"].mean():.2f}x')

# добавляем в бенчмарк для сравнения моделей
benchmark['MLP_classifier'] = [
    mlp_df['ROC-AUC'].mean(),
    mlp_df['PR-AUC'].mean(),
    mlp_df['PR/baseline'].mean()
]

MLP-классификатор (тот же DL, но без графовой структуры)

Fold: hold-out = Москва
    Epoch  20: loss=0.4655 ROC=0.7711 PR=0.8006 (best PR=0.8121)
    Early stop на эпохе 22
  -> ROC=0.7846 PR=0.8121 PR/base=1.47x (stopped at 22)

Fold: hold-out = Санкт-Петербург
    Epoch  20: loss=0.4592 ROC=0.7678 PR=0.7704 (best PR=0.7968)
    Early stop на эпохе 22
  -> ROC=0.7911 PR=0.7968 PR/base=1.50x (stopped at 22)

Fold: hold-out = Нижний Новгород
    Epoch  20: loss=0.4611 ROC=0.8011 PR=0.7551 (best PR=0.7593)
    Early stop на эпохе 24
  -> ROC=0.7997 PR=0.7593 PR/base=1.69x (stopped at 24)

Fold: hold-out = Новосибирск
    Epoch  20: loss=0.4592 ROC=0.8505 PR=0.8039 (best PR=0.8171)
    Early stop на эпохе 22
  -> ROC=0.8600 PR=0.8171 PR/base=1.84x (stopped at 22)

Fold: hold-out = Казань
    Epoch  20: loss=0.4643 ROC=0.7793 PR=0.7796 (best PR=0.7907)
    Early stop на эпохе 20
  -> ROC=0.7837 PR=0.7907 PR/base=1.56x (stopped at 20)

MLP СРЕДНЕЕ:
  ROC-AUC:     0.8038
  PR-AUC:      0.79

**финал**

In [24]:
benchmark_df = pd.DataFrame(benchmark, index=['ROC-AUC', 'PR-AUC', 'PR/baseline']).T
benchmark_df = benchmark_df.sort_values('PR-AUC', ascending=False)
benchmark_df

,ROC-AUC,PR-AUC,PR/baseline
CatBoost_baseline,0.873300,0.869800,1.760000
CatBoost_+_GNN_embeddings,0.870136,0.867649,1.759750
CatBoost_+_contrastive_embeddings,0.871300,0.867600,1.760000
MLP_classifier,0.803829,0.795206,1.611599
GraphSAGE_end_to_end,0.744127,0.749136,1.510552


На текущих 95 табличных признаках достигнут потолок задачи около PR-AUC 0.87 / PR/baseline 1.76x, и ни одна из протестированных DL-моделей (Spatial Contrastive Encoder, GraphSAGE end-to-end, CatBoost + GraphSAGE-эмбеддинги, MLP) не смогла его пробить. Это говорит о том, что проблема не в способе обработки имеющихся фичей, а в их информативности - часть размещений банкоматов объясняется не характеристиками района, а историческими и бизнес-факторами, которых в данных нет. Для дальнейшего улучшения метрик нужно обогащение датасета новыми источниками